In [3]:
import config as cfg
from data_acquirer import get_data, gen_data
from data_processor import process
import helpers
import heuristic
from heuristic import Path
from solver import solve_pdpcr

import copy
import datetime
from docplex.mp.model import Model
import logging
import heapq
import numpy as np
import time

logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

In [ ]:
num_robots = np.random.randint(1,[2,3,4,5,6,7,8,9,10,11],[10,10])

num_robots_fixed = np.array([[ 1,  2,  3,  1,  1,  2,  3,  3,  2,  9],
       [ 1,  1,  1,  1,  1,  1,  3,  4,  8,  8],
       [ 1,  2,  2,  1,  2,  3,  5,  4,  4, 10],
       [ 1,  1,  3,  4,  1,  2,  7,  1,  6,  6],
       [ 1,  1,  1,  3,  5,  2,  3,  6,  7,  7],
       [ 1,  1,  3,  2,  1,  5,  5,  4,  4,  9],
       [ 1,  2,  3,  2,  4,  1,  4,  7,  7,  6],
       [ 1,  1,  3,  4,  3,  6,  3,  2,  8,  4],
       [ 1,  2,  1,  1,  5,  3,  3,  4,  4,  9],
       [ 1,  2,  2,  3,  4,  4,  7,  5,  9,  7]])

num_robots_fixed2 = np.array([[1, 1, 2, 1, 5, 1, 7, 7, 4, 3],
       [1, 2, 1, 3, 5, 3, 3, 2, 4, 4],
       [1, 1, 2, 1, 4, 1, 4, 4, 1, 7],
       [1, 2, 3, 3, 2, 5, 5, 8, 9, 8],
       [1, 1, 3, 4, 4, 6, 6, 3, 6, 4],
       [1, 1, 1, 4, 1, 2, 5, 2, 5, 5],
       [1, 1, 2, 1, 5, 6, 1, 6, 4, 1],
       [1, 1, 1, 4, 2, 2, 3, 5, 6, 5],
       [1, 2, 1, 4, 4, 2, 2, 4, 4, 6],
       [1, 2, 1, 3, 1, 2, 5, 8, 9, 4]])

In [ ]:
# CPLEX data generated
data_idcs = [4,5,16,18]
data_paths = [f'generated{i}.csv' for i in data_idcs]
num_robots = [10]
file_type = 'csv'
i = 0
cfg.SOLVE_TIME =900
is_depot_task = False
for data_path in data_paths:
    for num_robot in num_robots:
        for i in range(1):
            try:
                # cur_num_robots = []
                # cur_num_robots = num_robots_fixed2[:,i]
        

                raw = get_data(data_path, file_type, is_depot_task)
                avg_num_robots = np.mean([raw.tasks[i].n for i in range(len(raw.tasks))])
                print(f'Solving for {data_path} with {num_robot} robots with average n_j = {avg_num_robots}')
                method = 5
                processed_data_vrp = process(raw, method)
                solve_result_vrp = solve_pdpcr(processed_data_vrp, num_robot, method)
                # method = 2
                # processed_data_flow_old = process(raw, method)
                # solve_result_flow_old = solve_pdpcr(processed_data_flow_old, num_robot, method)
                

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)

                # solve_log.append(solve_result_flow_old['solve_time'])
                # solve_log.append(solve_result_flow_old['sol'].objective_value)
                # solve_log.append(solve_result_flow_old['processed_nodes'])
                # solve_log.append(solve_result_flow_old['gap'])

                solve_log.append(solve_result_vrp['solve_time'])
                solve_log.append(solve_result_vrp['sol'].objective_value)
                solve_log.append(solve_result_vrp['processed_nodes'])
                solve_log.append(solve_result_vrp['gap'])

                now = datetime.datetime.now()
                solve_log.append(avg_num_robots)
                solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
                with open('solve_results/solve_log_li_lim_vrp.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)
            i += 1

In [ ]:
# read file 'solve_results/solve_log_generated_progressive_large_nj.csv' to pandas dataframe, it has the following columns:
# data_name, total_robots, solve_time_vrp, obj_value_vrp, processed_nodes_vrp, gap_vrp, solve_time_flow, obj_value_flow, processed_nodes_flow, gap_flow, avg_num_robots,date
import pandas as pd
df_large_nj = pd.read_csv('solve_results/solve_log_generated_progressive_large_nj.csv', header=None)
df_large_nj.columns = ['data_name', 'total_robots', 'solve_time_vrp', 'obj_value_vrp', 'processed_nodes_vrp', 'gap_vrp', 'solve_time_flow', 'obj_value_flow', 'processed_nodes_flow', 'gap_flow','avg_num_robots', 'date']


In [ ]:
first_names = ['lc','lr','lrc']
sec_names = [i for i in range(101,115)] +[i for i in range(201,215)]
# third_names = [i for i in range(1,11)]
data_paths =[f'lim/100/{i}{j}-5-uniform.csv'for i in first_names for j in sec_names]#
num_robots = [10*i for i in range(1,2)]
file_type = 'csv'
is_depot_task = False

In [ ]:
MAX_TL_ADD_SIZE = int(len(tasks)/4)
NUM_REROUTE_NEIGHBORS =   int(len(tasks)/3)
NUM_REVERSE_NEIGHBORS =  int(len(tasks)/9)
NUM_REDISTRIBUTE_NEIGHBORS = 0 #int(len(tasks)/15)
MAX_ITER = 5000
FIND_FROM_ELITE_AFTER = 200 #iterations
RESTART_AFTER = 400 #iterations
REVERSE_ON_CRITICAL = True
REROUTE_ON_CRITICAL = True
REROUTE_LONGEST_EDGE = True

In [ ]:
data_paths = ['lim/200/LC1_2_3-5-uniform.csv']
init_solution_path = f'solve_results/greedy2/LC1_2_3-5-uniform_2024-11-13_16-06-46'

In [ ]:
sol_fetcher = heuristic.SolutionFetcher(init_solution_path)
init_sol = sol_fetcher.fetch()

In [ ]:
edge_weight = {edge: init_sol.edges[edge]['weight'] for edge in init_sol.edges}

#get total out weight of each node
out_weight = {node: sum([edge_weight[edge] for edge in init_sol.out_edges(node)]) for node in init_sol.nodes}
#get total in weight of each node
in_weight = {node: sum([edge_weight[edge] for edge in init_sol.in_edges(node)]) for node in init_sol.nodes}

In [ ]:
print(in_weight)

In [22]:
#Li larger data

first_names = ['LC1']
sec_names = [10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [23]:
## Mass solve using tabu_search
# data_idcs = [16,18,19]
# data_paths = [f'generated{i}.csv' for i in data_idcs]
num_robots = [5]
file_type = 'csv'
is_depot_task = False
note = '3 neighborhoods'

for data_path in data_paths:
    for num_robot in num_robots:   
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/6)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/4)
            cfg.TABU_TL_REDISTRIBUTE_SIZE = int(len(tasks)/15)
            # cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/10)
            # cfg.TABU_NUM_REVERSE_NEIGHBORS =  int(len(tasks)/15)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/8)
            cfg.TABU_NUM_REVERSE_NEIGHBORS =  int(len(tasks)/15)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = int(len(tasks)/15)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 5000
            cfg.TABU_FIND_FROM_ELITE_AFTER = 300 #iterations
            cfg.TABU_RESTART_AFTER = 600 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            # init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()  
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate) 
                # heuristic.SolutionFetcher, 
                # init_solution_path = init_solution_path)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER,cfg.TABU_MAX_TL_ADD_SIZE)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%d-%m-%Y")
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            solve_log.append(note)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

26-May-25 10:42:19:INFO:Solving data lim/1000/LC1_10_1-5-uniform.csv with 5 robots and 527 tasks
26-May-25 10:42:21:INFO:TABU SEARCH PARAMETERS
26-May-25 10:42:21:INFO:RANDOM_SEED: 16
26-May-25 10:42:21:INFO:NUM_REVERSE_NEIGHBORS: 35
26-May-25 10:42:21:INFO:NUM_REROUTE_NEIGHBORS: 65
26-May-25 10:42:21:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 35
26-May-25 10:42:21:INFO:MAX_TL_ADD_SIZE: 87
26-May-25 10:42:21:INFO:MAX_TL_DROP_SIZE: 131
26-May-25 10:42:21:INFO:MAX_TL_REDISTRIBUTE_SIZE: 35
26-May-25 10:42:21:INFO:MAX_ITER: 5000
26-May-25 10:42:21:INFO:FIND_FROM_ELITE_AFTER: 300
26-May-25 10:42:21:INFO:RESTART_AFTER: 600
26-May-25 10:42:21:INFO:REROUTE_ON_CRITICAL: True
26-May-25 10:42:21:INFO:REVERSE_ON_CRITICAL: True
26-May-25 10:42:21:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True


Viet generalized sol is 11291.01
Viet generalized sol is 11438.29
Viet generalized sol is 11552.56
Viet generalized sol is 11815.65
Viet generalized sol is 11298.14
Viet generalized sol is 12249.11
Viet generalized sol is 11890.29
Viet generalized sol is 11905.91
Viet generalized sol is 12252.82
Viet generalized sol is 12359.97
Viet generalized sol is 11203.93
Viet generalized sol is 11478.19
Viet generalized sol is 11519.44
Viet generalized sol is 11739.48
Viet generalized sol is 11756.98
Viet generalized sol is 12392.92
Viet generalized sol is 11788.91
Viet generalized sol is 12362.56
Viet generalized sol is 11758.51
Viet generalized sol is 11451.67
Viet generalized sol is 11995.43
Viet generalized sol is 12015.77
Viet generalized sol is 11787.0
Viet generalized sol is 11490.82
Viet generalized sol is 11650.96
Viet generalized sol is 11748.52
Viet generalized sol is 11932.17
Viet generalized sol is 12237.47
Viet generalized sol is 11633.55


26-May-25 10:43:27:INFO:Initial makespan: 11203.92934742444
26-May-25 10:43:27:INFO:Initial fitness: 11203.92934742444


Viet generalized sol is 11241.72


26-May-25 10:43:30:INFO:Best fitness updated: 11197.04448168298, chosen neighborhood: 0
26-May-25 10:43:30:INFO:Iteration 0 done
26-May-25 10:43:32:INFO:Best fitness updated: 11194.669280227254, chosen neighborhood: 0
26-May-25 10:43:35:INFO:Best fitness updated: 11192.479654852159, chosen neighborhood: 0
26-May-25 10:43:42:INFO:Best fitness updated: 11187.301802293465, chosen neighborhood: 0
26-May-25 10:43:45:INFO:Best fitness updated: 11152.856781167133, chosen neighborhood: 0
26-May-25 10:43:49:INFO:Best fitness updated: 11149.399600675657, chosen neighborhood: 0
26-May-25 10:43:52:INFO:Best fitness updated: 11143.620655112205, chosen neighborhood: 0
26-May-25 10:43:55:INFO:Best fitness updated: 11134.004433679554, chosen neighborhood: 0
26-May-25 10:43:59:INFO:Best fitness updated: 11131.566749759215, chosen neighborhood: 0
26-May-25 10:44:03:INFO:Best fitness updated: 11129.833635600404, chosen neighborhood: 0
26-May-25 10:44:09:INFO:Best fitness updated: 11128.276728718007, chos

Tabu time: 15821.741315364838


... (2 outputs left out)

26-May-25 15:07:06:INFO:Initial makespan: 16963.161037392718
26-May-25 15:07:06:INFO:Initial fitness: 16963.161037392718


Viet generalized sol is 17596.26


26-May-25 15:07:09:INFO:Best fitness updated: 16962.818501730122, chosen neighborhood: 0
26-May-25 15:07:09:INFO:Iteration 0 done
26-May-25 15:07:12:INFO:Best fitness updated: 16957.759163274946, chosen neighborhood: 0
26-May-25 15:07:15:INFO:Best fitness updated: 16952.54689518911, chosen neighborhood: 0
26-May-25 15:07:18:INFO:Best fitness updated: 16950.263576124617, chosen neighborhood: 0
26-May-25 15:07:21:INFO:Best fitness updated: 16946.85703845909, chosen neighborhood: 0
26-May-25 15:07:24:INFO:Best fitness updated: 16941.57992186852, chosen neighborhood: 0
26-May-25 15:07:31:INFO:Best fitness updated: 16938.967034090467, chosen neighborhood: 0
26-May-25 15:07:34:INFO:Best fitness updated: 16935.032305187906, chosen neighborhood: 0
26-May-25 15:07:40:INFO:Best fitness updated: 16931.54260776859, chosen neighborhood: 0
26-May-25 15:07:43:INFO:Best fitness updated: 16930.99764648098, chosen neighborhood: 1
26-May-25 15:07:46:INFO:Best fitness updated: 16926.798646364437, chosen n

KeyboardInterrupt: 

In [ ]:
## Mass solve using tabu_search
# data_idcs = [16,18,19]
# data_paths = [f'generated{i}.csv' for i in data_idcs]
num_robots = [5]
file_type = 'csv'
is_depot_task = False
note = '2 neighborhoods'

for data_path in data_paths:
    for num_robot in num_robots:   
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/4)
            cfg.TABU_TL_REDISTRIBUTE_SIZE = int(len(tasks)/15)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/3)
            cfg.TABU_NUM_REVERSE_NEIGHBORS =  int(len(tasks)/9)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS =0# int(len(tasks)/15)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 5000
            cfg.TABU_FIND_FROM_ELITE_AFTER = 300 #iterations
            cfg.TABU_RESTART_AFTER = 600 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            # init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()  
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate) 
                # heuristic.SolutionFetcher, 
                # init_solution_path = init_solution_path)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER,cfg.TABU_MAX_TL_ADD_SIZE)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%d-%m-%Y")
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            solve_log.append(note)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

23-May-25 17:56:33:INFO:Solving data lim/800/LC1_8_1-5-uniform.csv with 20 robots and 420 tasks
23-May-25 17:56:34:INFO:TABU SEARCH PARAMETERS
23-May-25 17:56:34:INFO:RANDOM_SEED: 16
23-May-25 17:56:34:INFO:NUM_REVERSE_NEIGHBORS: 46
23-May-25 17:56:34:INFO:NUM_REROUTE_NEIGHBORS: 140
23-May-25 17:56:34:INFO:NUM_REDISTRIBUTE_NEIGHBORS: 0
23-May-25 17:56:34:INFO:MAX_TL_ADD_SIZE: 52
23-May-25 17:56:34:INFO:MAX_TL_DROP_SIZE: 105
23-May-25 17:56:34:INFO:MAX_TL_REDISTRIBUTE_SIZE: 28
23-May-25 17:56:34:INFO:MAX_ITER: 5000
23-May-25 17:56:34:INFO:FIND_FROM_ELITE_AFTER: 300
23-May-25 17:56:34:INFO:RESTART_AFTER: 600
23-May-25 17:56:34:INFO:REROUTE_ON_CRITICAL: True
23-May-25 17:56:34:INFO:REVERSE_ON_CRITICAL: True
23-May-25 17:56:34:INFO:REVERSE_CRITICAL_LONGEST_EDGE: True


Viet generalized sol is 2464.62
Viet generalized sol is 2572.38
Viet generalized sol is 2719.42
Viet generalized sol is 2787.05
Viet generalized sol is 2560.99
Viet generalized sol is 2769.66
Viet generalized sol is 2573.73
Viet generalized sol is 2530.79
Viet generalized sol is 2697.11
Viet generalized sol is 2754.66
Viet generalized sol is 2607.0
Viet generalized sol is 2627.24
Viet generalized sol is 2513.05
Viet generalized sol is 2748.46
Viet generalized sol is 2707.72
Viet generalized sol is 2802.1
Viet generalized sol is 2811.69
Viet generalized sol is 2740.59
Viet generalized sol is 2665.53
Viet generalized sol is 2713.54
Viet generalized sol is 2813.77
Viet generalized sol is 2709.77
Viet generalized sol is 2705.57
Viet generalized sol is 2735.43
Viet generalized sol is 3017.54
Viet generalized sol is 2668.79
Viet generalized sol is 2693.43
Viet generalized sol is 2745.72
Viet generalized sol is 2713.77


23-May-25 17:57:47:INFO:Initial makespan: 2464.617494859905
23-May-25 17:57:47:INFO:Initial fitness: 2464.617494859905


Viet generalized sol is 2709.81


23-May-25 17:57:49:INFO:Best fitness updated: 2463.4504905226745, chosen neighborhood: 0
23-May-25 17:57:49:INFO:Iteration 0 done
23-May-25 17:57:58:INFO:Best fitness updated: 2453.736759651032, chosen neighborhood: 0
23-May-25 17:58:05:INFO:Best fitness updated: 2453.5595918603085, chosen neighborhood: 0
23-May-25 17:58:08:INFO:Best fitness updated: 2453.3677500544027, chosen neighborhood: 0
23-May-25 17:58:09:INFO:Best fitness updated: 2451.7472325835784, chosen neighborhood: 0
23-May-25 17:58:13:INFO:Best fitness updated: 2451.38833326389, chosen neighborhood: 0
23-May-25 17:58:26:INFO:Best fitness updated: 2451.2173096178735, chosen neighborhood: 0
23-May-25 17:58:39:INFO:Best fitness updated: 2450.036136331507, chosen neighborhood: 0
23-May-25 17:58:41:INFO:Best fitness updated: 2447.495809219632, chosen neighborhood: 0
23-May-25 17:58:44:INFO:Best fitness updated: 2444.8138746056584, chosen neighborhood: 0
23-May-25 17:58:52:INFO:Best fitness updated: 2444.1436227796244, chosen n

Tabu time: 7301.269000530243


... (60 outputs left out)

24-May-25 12:04:12:INFO:Initial makespan: 3165.1815820451907
24-May-25 12:04:12:INFO:Initial fitness: 3165.1815820451907


Viet generalized sol is 3359.53


24-May-25 12:04:13:INFO:Iteration 0 done
24-May-25 12:04:21:INFO:Best fitness updated: 3165.1319704308785, chosen neighborhood: 0
24-May-25 12:04:23:INFO:Best fitness updated: 3164.791699819091, chosen neighborhood: 0
24-May-25 12:04:28:INFO:Best fitness updated: 3164.118465567885, chosen neighborhood: 0
24-May-25 12:04:31:INFO:Best fitness updated: 3163.8736626370614, chosen neighborhood: 0
24-May-25 12:04:32:INFO:Best fitness updated: 3162.1536149345397, chosen neighborhood: 0
24-May-25 12:04:34:INFO:Best fitness updated: 3158.8802859460684, chosen neighborhood: 0
24-May-25 12:04:35:INFO:Best fitness updated: 3157.8944643408986, chosen neighborhood: 0
24-May-25 12:04:45:INFO:Best fitness updated: 3156.8187551317124, chosen neighborhood: 0
24-May-25 12:04:49:INFO:Best fitness updated: 3153.0784765678673, chosen neighborhood: 0
24-May-25 12:04:52:INFO:Best fitness updated: 3153.0345641324548, chosen neighborhood: 0
24-May-25 12:05:08:INFO:Best fitness updated: 3150.7734477628765, chose

Tabu time: 7471.786859750748


In [ ]:
## Mass solve using tabu_search
#ONLY REDISTRIBUTE
num_robots = [5]
file_type = 'csv'
is_depot_task = False

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/8)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  0 #int(len(tasks)/5)
            cfg.TABU_NUM_REVERSE_NEIGHBORS = 0 # int(len(tasks)/10)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = int(len(tasks)/20)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            cfg.TABU_MAX_ITER = 200
            cfg.TABU_FIND_FROM_ELITE_AFTER = 200 #iterations
            cfg.TABU_RESTART_AFTER = 400 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d") + 'redis_only'
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve using tabu_search
#ONLY REROUTE
num_robots = [10]
file_type = 'csv'
is_depot_task = False

for num_robot in num_robots:
    for data_path in data_paths:
        try:
            raw = get_data(data_path, file_type, is_depot_task)
            logging.info(f'Solving data {data_path} with {num_robot} robots and {len(raw.tasks)} tasks')
            # processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            cfg.TABU_MAX_TL_ADD_SIZE = int(len(tasks)/8)
            cfg.TABU_MAX_TL_DROP_SIZE = int(len(tasks)/8)
            cfg.TABU_NUM_REROUTE_NEIGHBORS =  int(len(tasks)/7)
            cfg.TABU_NUM_REVERSE_NEIGHBORS = 0 # int(len(tasks)/10)
            cfg.TABU_NUM_REDISTRIBUTE_NEIGHBORS = 0 #int(len(tasks)/20)
            # cfg.TABU_MAX_TL_ADD_SIZE = 12
            # cfg.TABU_MAX_TL_DROP_SIZE = 12
            # cfg.TABU_NUM_REROUTE_NEIGHBORS = 20
            # cfg.TABU_NUM_REVERSE_NEIGHBORS = 10
            cfg.TABU_MAX_ITER = 200
            cfg.TABU_FIND_FROM_ELITE_AFTER = 200 #iterations
            cfg.TABU_RESTART_AFTER = 400 #iterations
            cfg.TABU_REVERSE_ON_CRITICAL = True
            cfg.TABU_REROUTE_ON_CRITICAL = True
            cfg.TABU_REROUTE_LONGEST_EDGE = True
            cfg.TABU_REDISTRIBUTE_PROB = 0.3
            cfg.TABU_SAVE_FITNESSES = True
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)

            data_name = data_path.split('/')[2].split('.')[0]
            init_solution_path = f'solve_results/init_data/{data_name}' #For fetch initial solution
            tik1 = time.time()
            tabu = heuristic.TabuSearch(
                processed_h, 
                c_travel, 
                c_operate, 
                heuristic.SolutionFetcher, 
                init_solution_path = init_solution_path)
            
            best_sol, best_sol_obj = tabu.search(num_robot,cfg.TABU_MAX_ITER)
            tok1 = time.time()
            tabu_time = tok1 - tik1
            print(f'Tabu time: {tabu_time}')
            if cfg.TABU_SAVE_FITNESSES:
                data_name = data_path.split('/')[2].split('.')[0]
                #add current time to data_name
                data_name += '_'+ datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + 'reroute_only'
                tabu.save_fitness_progress(data_name)
        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(best_sol_obj)
            solve_log.append(np.round(tabu_time,2))
            now = datetime.datetime.now()
            solve_log.append(now)
            with open('solve_log_tabu.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
cProfile.run(tabu.search(num_robot,cfg.TABU_MAX_ITER))

In [ ]:
## mass solve using heuristics
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_data = process(raw, method)
            processed_h = heuristic.process_greedy_1(raw)
            tik1 = time.time()
            solve_result_h = heuristic.solve_greedy_1(processed_h, num_robot)
            tok1 = time.time()
            greedy1_time = tok1 - tik1
            print(f'Greedy 1 time: {greedy1_time}')
            tik2 = time.time()
            solve_result_h_jp = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_jp_time = tok2 -tik2
            print(f'Greedy jp time: {greedy_jp_time}')
            tik3 = time.time()
            solve_result_h_jp_2 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3
            print(f'Greedy jp time 2: {greedy_jp_time_2}')

        
            solve_log = []
            solve_log.append(data_path)
            solve_log.append(solve_result_h.get_makespan().round(2))
            solve_log.append(np.round(greedy1_time,2))
            solve_log.append(solve_result_h_jp.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time,2))
            solve_log.append(solve_result_h_jp_2.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            with open('solve_log_h_2.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
import helpers
## Mass solve 2 opt
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)

            tik2 = time.time()
            solve_result_h2 = heuristic.solve_greedy_jp(processed_h, num_robot)
            tok2 = time.time()
            greedy_h2 = tok2 -tik2

            tik3 = time.time()
            solve_result_h3 = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
            tok3 = time.time()
            greedy_jp_time_2 = tok3 -tik3

            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)
            graph,makespan_2opt = solve_opt.solve(total_robots= num_robot)
            tok = time.time()
            two_opt_time = tok - tik

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_result_h2.get_makespan().round(2))
            solve_log.append(np.round(greedy_h2,2))
            solve_log.append(solve_result_h3.get_makespan().round(2))
            solve_log.append(np.round(greedy_jp_time_2,2))
            solve_log.append(makespan_2opt.round(2))
            solve_log.append(np.round(two_opt_time,2))
            now = datetime.datetime.now()
            solve_log.append(now.strftime("%Y-%m-%d %H:%M:%S"))
            with open('solve_log_h_2opt.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)

In [ ]:
## Mass solve Viet 

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
consider_process_time =True
for data_path in data_paths:
    for num_robot in num_robots:
        try:
            print(f'Solving for {data_path} with {num_robot} robots')
            raw = get_data(data_path, file_type, is_depot_task)
            processed_h = heuristic.process_greedy_1(raw)
            depot = processed_h.depot
            tasks = processed_h.tasks
            c_travel = helpers.create_task_dist_dict(depot,tasks)
            c_operate = helpers.create_operate_dict(depot,tasks)
            tik = time.time()
            solve_viet = heuristic.VietSolverOrig(processed_h, c_travel, c_operate)
            solve_viet.solve(num_robot,consider_process_time)
            tok = time.time()
            time_viet = tok - tik
            print(f'makespan: {solve_viet.get_makespan()}')
            print('Done')

            solve_log = []
            solve_log.append(data_path)
            solve_log.append(num_robot)
            solve_log.append(solve_viet.get_makespan().round(2))
            solve_log.append(np.round(time_viet,2))
            with open('solve_log_h_Viet_09-04-process-time.csv', 'a') as f:
                f.write(','.join([str(i) for i in solve_log])+'\n')
        except Exception as e:
            print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
            print(e)


In [ ]:
## Mass solve Viet k tasks

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')

tasks_each_iters = [5]
consider_process_time =False
for task_each_ite in tasks_each_iters:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)
                tik = time.time()
                solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                sol = solve_viet.solve(num_robot,task_each_ite,consider_process_time)
                tok = time.time()
                time_viet = tok - tik
                makespan = sol.get_makespan()
                print(f'makespan: {makespan}')
                print('Done')

                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(task_each_ite)
                solve_log.append(makespan.round(2))
                solve_log.append(np.round(time_viet,2))
                with open('solve_log_h_Viet k task large instances 23-04.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


In [ ]:
a ={0: 0.0, 1: 599.9817763899168, 2: 271.90904527065663, 3: 283.1551518161024, 4: 142.61641735789044, 5: 239.53321272842314, 6: 503.8917411508151, 7: 320.3896980241406, 8: 347.00099826945745, 9: 177.80785134520917, 10: 194.39276118209756, -1: 0}

In [13]:
#Li larger data

first_names = ['LC1']
sec_names = [2,4,6,8,10]
third_names = [i for i in range(1,11)]
data_paths =[f'lim/{j}00/{i}_{j}_{k}-5-uniform.csv'for i in first_names for j in sec_names for k in third_names]#
num_robots = [10*i for i in range(1,4)]
method = 5
file_type = 'csv'
is_depot_task = False

In [ ]:
## Mass solve Viet or jp k tasks with random k

import logging
logging.basicConfig(format=cfg.LOG_FORMATTER, level=cfg.LOG_LEVEL, datefmt='%d-%b-%y %H:%M:%S')
num_robots = [5]
ite = 50
tasks_each_iter_ranges = [[2,4]]
note = 'JP_solver 80 15 5'
# tasks_each_iter_ranges = [[1,2]]
consider_process_time =False
for task_each_ite in tasks_each_iter_ranges:
    for data_path in data_paths:
        for num_robot in num_robots:
            try:
                print(f'Solving for {data_path} with {num_robot} robots')
                raw = get_data(data_path, file_type, is_depot_task)
                processed_h = heuristic.process_greedy_1(raw)
                depot = processed_h.depot
                tasks = processed_h.tasks
                c_travel = helpers.create_task_dist_dict(depot,tasks)
                c_operate = helpers.create_operate_dict(depot,tasks)

                solve_result_h_jp = heuristic.solve_greedy_jp_compare_start_time(processed_h, num_robot)
                TT_makespan = solve_result_h_jp.get_makespan()

                tik = time.time()
                cur_best = np.infty
                cur_worst = -np.infty
                makespan_results = []
                no_better_TT = 0

                for i in range(ite):
                    
                    # solve_viet = heuristic.JPGeneralizedSolver(processed_h, c_travel, c_operate)
                    solve_viet = heuristic.VGeneralizedSolver(processed_h, c_travel, c_operate)
                    sol = solve_viet.solve(num_robot,'random',consider_process_time,task_range=task_each_ite)
                    makespan = sol.get_makespan()
                    makespan_results.append(makespan)
                    if makespan < cur_best:
                        cur_best = makespan
                    if makespan > cur_worst:
                        cur_worst = makespan
                    if makespan <= TT_makespan:
                        no_better_TT +=1
                avg_makespan = np.mean(makespan_results)
                tok = time.time()
                cpu = tok - tik
            
                solve_log = []
                solve_log.append(data_path)
                solve_log.append(num_robot)
                solve_log.append(ite)
                solve_log.append(cur_best.round(2))
                solve_log.append(cur_worst.round(2))
                solve_log.append(avg_makespan.round(2))
                solve_log.append(np.round(1-cur_best/TT_makespan,4)*100)
                solve_log.append(np.round(1-avg_makespan/TT_makespan,4)*100)
                solve_log.append(np.round(cpu,2))
                solve_log.append(no_better_TT)
                solve_log.append(note)
                with open('solve_log_h_JP large instances.csv', 'a') as f:
                    f.write(','.join([str(i) for i in solve_log])+'\n')
            except Exception as e:
                print(f'\n ______________Error in {data_path} with {num_robot} robots______________\n')
                print(e)


Solving for lim/200/LC1_2_1-5-uniform.csv with 5 robots
JP heuristics sol is 1551.57
Viet generalized sol is 1643.98
Viet generalized sol is 1849.93
Viet generalized sol is 1924.46
Viet generalized sol is 1716.13
Viet generalized sol is 1993.52
Viet generalized sol is 1697.34
Viet generalized sol is 1983.04
Viet generalized sol is 1557.04
Viet generalized sol is 2061.79
Viet generalized sol is 1869.41
Viet generalized sol is 1722.89
Viet generalized sol is 1798.84
Viet generalized sol is 1695.68
Viet generalized sol is 1718.83
Viet generalized sol is 1518.54
Viet generalized sol is 1642.76
Viet generalized sol is 1589.12
Viet generalized sol is 1778.94
Viet generalized sol is 1657.97
Viet generalized sol is 1657.87
Viet generalized sol is 1733.73
Viet generalized sol is 1734.29
Viet generalized sol is 1633.24
Viet generalized sol is 1671.09
Viet generalized sol is 1760.78
Viet generalized sol is 1669.17
Viet generalized sol is 1513.32
Viet generalized sol is 1684.9
Viet generalized sol

In [ ]:
raw = get_data(data_path, file_type, is_depot_task)
processed_h = heuristic.process_greedy_1(raw)
depot = processed_h.depot
tasks = processed_h.tasks
c_travel = helpers.create_task_dist_dict(depot,tasks)
c_operate = helpers.create_operate_dict(depot,tasks)
tik = time.time()
solve_opt = heuristic.TwoOptSolver(processed_h,c_travel, c_operate)

In [ ]:
#Get data

#Read file solve_log.csv, save it as a dataframe
import pandas as pd
solve_cplex_df = pd.read_csv('solve_log.csv', header=None)
solve_cplex_df.columns = ['data_path', 'num_robot', 'solve_time', 'objective_value', 'processed_nodes', 'gap', 'solve_date']

solve_h_df = pd.read_csv('solve_log_h.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_jp', 'time_greedy_jp']

In [ ]:
# right join 2 dataframes on column data_path
merged_df = pd.merge(solve_cplex_df, solve_h_df, on='data_path', how='right')

In [ ]:
merged_df['diff_h_1'] = np.round(1- merged_df['objective_value_1']/merged_df['objective_value'],2)
merged_df['diff_h_jp'] = np.round(1- merged_df['objective_value_jp']/merged_df['objective_value'],2)
#change order of columns in meged_df as follow: data_path, num_robot, objective_value, solve_time, objective_value_1, time_greedy_1, diff_h_1, objective_value_jp, time_greedy_jp, diff_h_jp,
merged_df = merged_df[['data_path', 'num_robot', 'objective_value', 'solve_time', 'objective_value_1', 'time_greedy_1', 'diff_h_1', 'objective_value_jp', 'time_greedy_jp', 'diff_h_jp']]
merged_df
#Remove the 'lim' prefix from data_path
merged_df['data_path'] = merged_df['data_path'].str.replace('lim/','')
#round solve_time to 2 decimal places
merged_df['solve_time'] = np.round(merged_df['solve_time'],2)
#round objective_value to 2 decimal places
merged_df['objective_value'] = np.round(merged_df['objective_value'],2)

In [ ]:
merged_df.to_latex('merged_df.tex', index=True)

In [ ]:
#Temp
import pandas as pd
solve_h_df = pd.read_csv('solve_log_h_2.csv', header=None)
solve_h_df.columns = ['data_path', 'objective_value_1', 'time_greedy_1', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3']
#Keep the following columns: data_path, objective_value_2, time_greedy_2
solve_h_df = solve_h_df[['data_path', 'objective_value_2', 'time_greedy_2']]

In [ ]:
solve_h_df_2opt = pd.read_csv('solve_log_h_2opt.csv', header=None)
solve_h_df_2opt.columns = ['data_path', 'num_robot', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']

In [ ]:
#Join 2 dataframes on data_path
merged_df_2 = pd.merge(solve_h_df, solve_h_df_2opt, on='data_path', how='right')
#Change the order of columns as follow: data_path, num_robot, objective_value_2, time_greedy_2, objective_value_3, time_greedy_3, objective_value_2opt, time_2opt
merged_df_2 = merged_df_2[['data_path', 'num_robot', 'objective_value_2', 'time_greedy_2', 'objective_value_3', 'time_greedy_3', 'objective_value_2opt', 'time_2opt']]

In [ ]:
#save the data as a latex table
merged_df_2.to_latex('merged_df_2.tex', index=True)